<a href="https://colab.research.google.com/github/d-noe/NLP_DH_F26/blob/main/code/1_introduction/Hands_on_1_tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hands-On Session 1: Tokenization

![](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fwww.moma.org%2Fd%2Fassets%2FW1siZiIsIjIwMjMvMDYvMjcvMTcweHg5ZHcxZ19JTjI0MTdfMDA1X0NDQ1JfRnVsbF9zaXplLmpwZyJdLFsicCIsImNvbnZlcnQiLCItcXVhbGl0eSA5MCAtcmVzaXplIDIwMDB4NjY3XiAtZ3Jhdml0eSBDZW50ZXIgLWNyb3AgMjAwMHg2NjcrMCswIl1d%2FIN2417_005_CCCR-Full-size.jpg%3Fsha%3Dfae4b38a360c0c32&f=1&nofb=1&ipt=f2ccb3b3225eaa9f848cb952197286fdfab98d8cefdc02878eac556ae0158170&ipo=images$0)
<p align="right">
  <i>Rhapsody (detail)</i>. Jennifer Bartlett. (1975–76). Enamel on steel. <br>© 2023 Jennifer Bartlett. Photograph: John Wronn
</p>




This notebook contains three section:
- [Tokenizers exploration](#exploration): use different 'off-the-shelf' tokenizers, play with the inputs and compare the outputs
- [BPE Implementation](#bpe): Implement you own version of the Byte-Pair Encoding algorithm and train you own tokenizer!
- [Zipf's Law](#zipf): Compute Zipf's law on real-world data (based on the tokenizer(s) of your choice, or even the one you trained!)

You can start by any of these sections.

<a name="exploration"></a>
# Tokenizers Exploration

Explore the tokenization results of different off-the-shelf tokenizers. By default (but feel free to test other methods), the following code includes tokenizers from:
- [`nltk`](https://www.nltk.org/)
- [`spaCy`](https://spacy.io/)
- [`tiktoken`](https://github.com/openai/tiktoken)(note: they also have an app if you want to directly [test it in browser](https://tiktokenizer.vercel.app/))

In [ ]:
# nltk
from nltk.tokenize import TreebankWordTokenizer
nltk_tokenizer = TreebankWordTokenizer()

# spacy
from spacy.lang.en import English
nlp = English()
spacy_tokenizer = nlp.tokenizer

# tiktoken
import tiktoken
tiktoken_tokenizer = tiktoken.get_encoding("gpt2") # we use the gpt-2 variant here, you can choose another model

In [ ]:
# build a dictionnary of tokenization methods to ease comparison

dict_tokenize = {
    "nltk.TreebankWordTokenizer": lambda text: nltk_tokenizer.tokenize(text),
    "spacy.English.tokenizer": lambda text: [tok for tok in spacy_tokenizer(text)], # decompose spacy output format
    "tiktoken.gpt-2_ids": lambda text: tiktoken_tokenizer.encode(text), # return integers IDs
    "tiktoken.gpt-2_tokens": lambda text: [
        tiktoken_tokenizer.decode([tok_id]) # decode to see the tokens
        for tok_id in tiktoken_tokenizer.encode(text) # return integers IDs
    ],
}

Here are a few sentences, play with your examples (different domains, language, alphabets; spelling errors, slang, neologism, inventions, etc.).

In [ ]:
sentences = [ # a few toy examples to test tokenization, feel free to add your own examples!
    "Times flies like an arrow; fruit flies like banana.",
    "Jean-Edouard n'aime pas l'IA.",
    "欲速则不达",
    "This cat is enooooormous.",
    "Notebook is weird. Weird is notebook",
]

In [ ]:
for s in sentences:
  print("-"*30)
  print(f"Original sentence: {s}")
  print("Tokenization:")
  for k, v in dict_tokenize.items():
    print(f"\t- {k}: {v(s)}")

- How does each tokenizer handle punctuation, accents, non-Latin scripts, ...?
- Count the total number of generated tokens per sentence across different libraries. Do you observe any patterns?
- Do the individual 'tokens' have semantic meaning? Why?
- Compare the number of tokens for different languages (including 'low-resource' languages), what do you observe (esp. when using BPE-like tokenization)? What are the potential implications (you can think in terms of computational cost and access to technology)?

Bonus:

- Compute the dependency tree for the famous sentence by Groucho Marx: "Times flies like an arrow; fruit flies like banana.": How does the model understand this sentence?

<a name="bpe"></a>
# BPE Implementation

Implement you own version of the BPE algorithm, you can check the details in the [original article](https://www.derczynski.com/papers/archive/BPE_Gage.pdf), or on [Wikipedia](https://en.wikipedia.org/wiki/Byte-pair_encoding).

If you do not want to completely start from scratch, a pre-filled skeleton of a `BPE` class is included below. If you use this code, you will have to implement the missing parts of the methods:
- the BPE-training loop in `train`, used to learn the merges to perform BPE
- the `_encode_word` logic used once the model is trained to apply the merge rules on given sequence of bytes, and get the token ids.

PS. In case you want to check / get inspiration, a proposed complete implementation is included [at the end of this notebook](#bpe_solution).

In [ ]:
from collections import Counter, defaultdict
import heapq
import regex


class BPE:
    def __init__(
        self,
        regex_pattern:str = r" ?\w+| ?\S+|\s+",
    ):
        """
        Initialise the BPE tokenizer with the 2**8 distinct byte values.

        Parameters:
            - regex_pattern [str | default: r" ?\w+| ?\S+|\s+"]
                Regular expression used to pre-tokenize text into categories
                to prevent BPE merges across natural boundary types.
                The default value can be decomposed as follow:
                    - ' ?\w+' matches optional space followed by letters/digit/underscores
                    - ' ?\S+' matches optional space followed by punctuations/symbols
                    - '\s+'   fallback to any remaining standalone whitesapce
                For reference, the regex used by gpt-2 tokenizer is: "'s|'t|'re|'ve|'m|'ll|'d| ?[\p{L}]+| ?[\p{N}]+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"

        """
        # ----------------- Vocabulary
        # Token ID -> bytes -- instantiated with the default 256 individual byte values
        self.vocab = {
            i: bytes([i])
            for i in range(256)
        }
        # Bytes -> token ID -- reverse 'self.vocab' dictionnary
        self.token_to_id = {
            token: i
            for i, token in self.vocab.items()
        }

        # ----------------- Merges
        # Complete list of learned merges (during training)
        self.merges = []
        # Merge rank (to speed up encoding, avoid full lookup)
        # (token_a, token_b) -> merge rank
        self.merge_rank = {}
        # (token_a, token_b) -> resulting token ID
        self.merge_token = {}

        # ----------------- Pre-tokenization rule
        self.regex_pattern = regex_pattern
        self._pat = regex.compile(regex_pattern)

    # ==========================
    # ~~~~~~~~ Training ~~~~~~~~
    # ==========================

    def train( # TODO: implement the missing parts
        self,
        corpus: str,
        vocab_size: int,
        verbose: bool = True
    ):
        """
        Train the BPE tokenizer on the given corpus:
            - learn byte-pair merges
            - expand vocabulary up to predetermined size
            - store merge priorities

        Parameters:
            - corpus [str]
                The textual corpus used to learn the merges.
            - vocab_size [int]
                The (maximal) size of the vocabulary.
                Note that vocab_size must be greater than 256 (number of individual bytes).
                The training algorithm will stop either
                    -> when the vocabulary is extended to vocab_size, or
                    -> when there are no more merges to do.
            - verbose [bool | default = True]
                Whether to print the iterative pair merges found during training or not.
        """

        if vocab_size < 256:
            raise ValueError(
                "vocab_size must be at least 256."
            )

        # ----------------- PRE-TRAINING SETUP -----------------
        # pre-tokenize the corpus based on regular expression into 'words'
        words = self._pat.findall(corpus)

        # turn characters to bytes for each 'word'
        data = [
            [bytes([b]) for b in word.encode("utf-8")]
            for word in words
        ]

        # Compute Initial Statistics
        pair_count = Counter()
        pair_words = defaultdict(set)
        for word_id, word in enumerate(data): # for each 'word'
            for pair in zip(word, word[1:]):  # iterate over consecutive bytes
                pair_count[pair] += 1         # count byte pairs
                pair_words[pair].add(word_id) # list of words containing pair

        # Priority queue (min-heap using negative counts) to rapidly query the most frequent pair
        # This format is used for efficiency, but it could be conceptually avoided
        heap = [
            (-count, pair)
            for pair, count in pair_count.items()
        ]
        heapq.heapify(heap)

        # -----------------  BPE TRAINING LOOP -----------------
        while len(self.vocab) < vocab_size: # iterate while reaching given vocab_size
            # TODO: Implement the training loop
            # 1. retrieve the current most frequent pair (a, b)
             # (be careful if complying with heapq, it has a negative count logic, check the documentation if needed)
             # 1'. Stop training if no more pairs to merge
            # 2. Define the new token (new_tok) and it's ID (new_id)
            # 3. Record the changes

            if verbose: # you should have all these variables now
                print(
                    f"{new_id:5d}: "
                    f"{a!r} + {b!r} -> {new_token!r} "
                    f"({count:,})"
                )

            # record the merge rule and register the new token in the vocabulary (+ reverse)
            self.merges.append(pair)
            self.vocab[new_id] = new_token
            self.token_to_id[new_token] = new_id

            # retrieve only the words that contain the target pair (for efficiency: avoids scanning the full corpus)
            affected = list(pair_words[pair])

            # TODO: update the representations
            # For each affected word (which contains the new merged token)
            # 3.
            for word_id in affected:
                word = data[word_id]

                # 1. Decrement counts for all adjacent pairs in the un-merged version

                # 2. Replace every occurence of consecutive (a, b) with the new token
                new_word = []
                i = 0
                while i < len(word):
                    # TODO: implement step 2.

                # update corpus with the newly merged structure: (a,b) -> new_token
                data[word_id] = new_word

                # 3. Increment count for the new adjacent paris after the introduction of the (a,b)-merge
                # increment count for the new adjacent pairs after the introduction of the (a,b)-merge
                new_pairs = list(zip(new_word, new_word[1:]))
                for p in new_pairs:
                    # TODO: implement step 3
                    pass

        # ----------------- POST-TRAINING LOOKUP SETUP -----------------
        # build fast lookup tables for encoding text
        self.merge_rank.clear()
        self.merge_token.clear()

        for rank, (a, b) in enumerate(self.merges): # in the order of the learned merges
            # map byte tokens to their assigned integer vocabulary IDs
            a_id = self.token_to_id[a]
            b_id = self.token_to_id[b]

            new_id = self.token_to_id[a + b]

            # priority rank: lower rank means the merge was learned earlier (higher precedence)
            self.merge_rank[(a_id, b_id)] = rank
            # map candidate token ID pairs to the resulting merged token ID
            self.merge_token[(a_id, b_id)] = new_id

    # ==========================
    # ~~~~~~~~ Encoding ~~~~~~~~
    # ==========================

    def encode(
        self,
        text: str,
    ):
        """
        Encode raw text into a sequence of BPE token IDs

        Steps:
        1. Pre-tokenize input text into 'words' using the regex pattern.
        2. Encode each word into bytes and map them to subword token IDs. (TODO: IMPELMENT THE METHOD)
        3. Flatten and concatenate the resulting token IDs into a single list.

        Parameters:
            - text [str]
                The string to be tokenized.

        Returns:
            - list[int]
                List of integer token IDs representing the tokenized text.
        """
        output = []

        for word in self._pat.findall(text):
            output.extend(
                self._encode_word( # TODO: you need to implement this method
                    word.encode("utf-8")
                )
            )

        return output

    def _encode_word(self, data: bytes):
        """
        Encode a single sequence of bytes into token IDs by iteratively applying
        the learned merges in order of priority.
        """
        if not data:
            return []

        # start with the individual byte token IDs
        tokens = [self.token_to_id[bytes([b])] for b in data]

        while len(tokens) >= 2:
            # TODO: implement the encoding loop!
            pass

        return tokens


    # ==========================
    # ~~~~~~~~ Decoding ~~~~~~~~
    # ==========================

    def decode(self, ids:list):
        """
        Turn list of IDs into tokens and decode UTF-8.
        """

        data = b"".join(
            self.vocab[i]
            for i in ids
        )

        return data.decode("utf-8")

Once implemented, you can train your own BPE tokenizer on the corpora of your choice. Feel free to play with the parameters: how does it impact the results? What happens when increasing/decreasing `n_vocab`? What is the impact of the pre-tokenizing regular expression pattern on the BPE dynamics?

In [ ]:
%%time

# Example use case to test your implementation
# Note: took approx. 50 seconds on the 'War and Peace' book with n_vocab=50257

try:
  corpus = txt # your corpus here!
except:
  corpus = "The cat is on the mat. The cat is sleeping on the mat. The cat is on the sofa."


bpe = BPE()

text = "the cat sat on the mat"

print("Before Training")

ids = bpe.encode(text)

print("\nTokens:")
print(ids)

print("\nDecoded:")
print([bpe.decode([tok]) for tok in ids])

print("\nTraining the BPE model...\n")

bpe.train(
    corpus,
    vocab_size=50257, # same vocabulary size as tiktoken's gpt2 model
)

print("\nAfter Training")

ids = bpe.encode(text)

print("\nTokens:")
print(ids)

print("\nDecoded:")
print([bpe.decode([tok]) for tok in ids])

<a name="zipf"></a>
# Zipf's Law

1. Download book(s) from the [Project Guteberg](https://www.gutenberg.org/) website (preferrably in `.txt` format), for instance:
  - [the complete works of Shakespeare](https://www.gutenberg.org/ebooks/100)
  - [Tolstoy's *War and Peace*](https://www.gutenberg.org/ebooks/2600)
2. Tokenize the text with the method(s!) of you choice
3. Compute and visualise the $f(r)$ plot

- Can you find the Zipf's law? (Challenge: find the empirical coefficient of the [Zipf-Mandlebrot's Law](https://en.wikipedia.org/wiki/Zipf%27s_law$) for your data).
- Do you observe any differences when using different tokenizers?
- What are the most/least frequent tokens?
- You can also try with different languages, what do you observe?

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter


In [ ]:
# Quick drafty helpers
def make_zipf_plot(
    sorted_counts_dict:dict={},
    log_plot:bool=True
):
    """
    Make the Zipf's plot
    Inputs:
      - sorted_counts_dict [dict]
        Format: {<METHOD_NAME> [str] : <COUNTS> [list]}
        A dictionnary with
          - keys: identifiers of the methods
          - values: sorted list of token counts
      - log_plot [bool:True]
        Whether to use a log-log scale on the plot.
    """
    fig, ax = plt.subplots(figsize=(10,10))
    for k, v in sorted_counts_dict.items():
        ax.plot(
            np.arange(len(v))+1,
            v,
            label=k,
        )

    ax.set_xlabel("Rank")
    ax.set_ylabel("Frequency")
    ax.legend()

    if log_plot:
        plt.loglog()
    plt.show()

def cumulative_freq_plot(
    sorted_counts_dict:dict,
    normalise_rank:bool=False
):
    fig, ax = plt.subplots(figsize=(10,10))
    for k, v in sorted_counts_dict.items():
        cumulative_sum = v.cumsum()
        cumulative_freq = cumulative_sum/cumulative_sum[-1]
        ax.plot(
            np.arange(len(cumulative_freq))+1 if not normalise_rank else (np.arange(len(cumulative_freq))+1)/len(cumulative_freq),
            cumulative_freq,
            label=k,
        )

    ax.set_xlabel("Rank" if not normalise_rank else "Normalised Rank")
    ax.set_ylabel("Cumulative Frequency")
    ax.legend()
    plt.plot()

In [ ]:
# load your input data

file_path = "pg79657.txt" # Replace with the path to your file

with open(os.path.join(file_path)) as f: # for a .txt file
    txt = f.read()
f.close()

print(f"Loaded file {file_path}: len = {len(txt):,}.")

In [ ]:
# TODO: tokenize the text

tokens = txt.split() # TODO: replace with the method you chose

In [ ]:
# count individual tokens

counter = Counter(tokens)

sorted_counts = np.fromiter(counter.values(), dtype=int)
sorted_counts.sort()
sorted_counts = sorted_counts[::-1]

print(f"The twenty most common tokens are: {counter.most_common(20)}.")

In [ ]:
# prepare format to comply with the helper function to make the plot

frequencies_dict = {
    "my_tokenization": sorted_counts
}

In [ ]:
# plot f(r)

make_zipf_plot(
    frequencies_dict
)

<a name="bpe_solution"></a>

**Proposed BPE Solution**

In [ ]:
from collections import Counter, defaultdict
import heapq
import regex


class BPE:
    def __init__(
        self,
        regex_pattern:str = r" ?\w+| ?\S+|\s+",
    ):
        """
        Initialise the BPE tokenizer with the 2**8 distinct byte values.

        Parameters:
            - regex_pattern [str | default: r" ?\w+| ?\S+|\s+"]
                Regular expression used to pre-tokenize text into categories
                to prevent BPE merges across natural boundary types.
                The default value can be decomposed as follow:
                    - ' ?\w+' matches optional space followed by letters/digit/underscores
                    - ' ?\S+' matches optional space followed by punctuations/symbols
                    - '\s+'   fallback to any remaining standalone whitesapce
                For reference, the regex used by gpt-2 tokenizer is: "'s|'t|'re|'ve|'m|'ll|'d| ?[\p{L}]+| ?[\p{N}]+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"

        """
        # ----------------- Vocabulary
        # Token ID -> bytes -- instantiated with the default 256 individual byte values
        self.vocab = {
            i: bytes([i])
            for i in range(256)
        }
        # Bytes -> token ID -- reverse 'self.vocab' dictionnary
        self.token_to_id = {
            token: i
            for i, token in self.vocab.items()
        }

        # ----------------- Merges
        # Complete list of learned merges (during training)
        self.merges = []
        # Merge rank (to speed up encoding, avoid full lookup)
        # (token_a, token_b) -> merge rank
        self.merge_rank = {}
        # (token_a, token_b) -> resulting token ID
        self.merge_token = {}

        # ----------------- Pre-tokenization rule
        self.regex_pattern = regex_pattern
        self._pat = regex.compile(regex_pattern)

    # ==========================
    # ~~~~~~~~ Training ~~~~~~~~
    # ==========================

    def train(
        self,
        corpus: str,
        vocab_size: int,
        verbose: bool = True
    ):
        """
        Train the BPE tokenizer on the given corpus:
            - learn byte-pair merges
            - expand vocabulary up to predetermined size
            - store merge priorities

        Parameters:
            - corpus [str]
                The textual corpus used to learn the merges.
            - vocab_size [int]
                The (maximal) size of the vocabulary.
                Note that vocab_size must be greater than 256 (number of individual bytes).
                The training algorithm will stop either
                    -> when the vocabulary is extended to vocab_size, or
                    -> when there are no more merges to do.
            - verbose [bool | default = True]
                Whether to print the iterative pair merges found during training or not.
        """

        if vocab_size < 256:
            raise ValueError(
                "vocab_size must be at least 256."
            )

        # ----------------- PRE-TRAINING SETUP -----------------
        # pre-tokenize the corpus based on regular expression into 'words'
        words = self._pat.findall(corpus)

        # turn characters to bytes for each 'word'
        data = [
            [bytes([b]) for b in word.encode("utf-8")]
            for word in words
        ]

        # Compute Initial Statistics
        pair_count = Counter()
        pair_words = defaultdict(set)
        for word_id, word in enumerate(data): # for each 'word'
            for pair in zip(word, word[1:]):  # iterate over consecutive bytes
                pair_count[pair] += 1         # count byte pairs
                pair_words[pair].add(word_id) # list of words containing pair

        # Priority queue (min-heap using negative counts) to rapidly query the most frequent pair
        # This format is used for efficiency, but it could be conceptually avoided
        heap = [
            (-count, pair)
            for pair, count in pair_count.items()
        ]
        heapq.heapify(heap)

        # -----------------  BPE TRAINING LOOP -----------------
        while len(self.vocab) < vocab_size: # iterate while reaching given vocab_size
            # Pop pairs from heap until finding one whose count matches the current actual frequency
            while heap:
                neg_count, pair = heapq.heappop(heap) # pop = get and remove
                count = -neg_count
                if pair_count.get(pair, 0) == count:
                    break
            else: # Stop training: heap is empty = no more pairs available to merge
                break
            if count < 2: # Stop training if the most frequent pair appears less than twice
                break

            # unpack most frequent pair and create its merged byte representation
            a, b = pair
            new_token = a + b
            new_id = len(self.vocab)

            if verbose:
                print(
                    f"{new_id:5d}: "
                    f"{a!r} + {b!r} -> {new_token!r} "
                    f"({count:,})"
                )

            # record the merge rule and register the new token in the vocabulary (+ reverse)
            self.merges.append(pair)
            self.vocab[new_id] = new_token
            self.token_to_id[new_token] = new_id

            # retrieve only the words that contain the target pair (for efficiency: avoids scanning the full corpus)
            affected = list(pair_words[pair])

            for word_id in affected: # for each affected word (= which contains the new merged token)
                word = data[word_id]

                # decrement counts for all adjacent pairs in the current un-merged word
                old_pairs = list(zip(word, word[1:]))
                for p in old_pairs:
                    pair_count[p] -= 1
                    pair_words[p].discard(word_id)

                # replace every occurence of consecutive sequence (a, b) with the new token
                new_word = []
                i = 0
                while i < len(word):
                    if (
                        i + 1 < len(word)
                        and word[i] == a
                        and word[i + 1] == b
                    ): # if we see the current pair
                        new_word.append(new_token) # replace (a,b) -> new_token
                        i += 2 # advance past both previous tokens
                    else: # if not the current pair
                        new_word.append(word[i]) # keep same token
                        i += 1 # advance to next token

                # update corpus with the newly merged structure: (a,b) -> new_token
                data[word_id] = new_word

                # increment count for the new adjacent pairs after the introduction of the (a,b)-merge
                new_pairs = list(zip(new_word, new_word[1:]))
                for p in new_pairs:
                    pair_count[p] += 1
                    pair_words[p].add(word_id)
                    # push updated pair frequency back to priority queue
                    heapq.heappush(
                        heap,
                        (-pair_count[p], p)
                    )

        # ----------------- POST-TRAINING LOOKUP SETUP -----------------
        # build fast lookup tables for encoding text
        self.merge_rank.clear()
        self.merge_token.clear()

        for rank, (a, b) in enumerate(self.merges): # in the order of the learned merges
            # map byte tokens to their assigned integer vocabulary IDs
            a_id = self.token_to_id[a]
            b_id = self.token_to_id[b]

            new_id = self.token_to_id[a + b]

            # priority rank: lower rank means the merge was learned earlier (higher precedence)
            self.merge_rank[(a_id, b_id)] = rank
            # map candidate token ID pairs to the resulting merged token ID
            self.merge_token[(a_id, b_id)] = new_id

    # ==========================
    # ~~~~~~~~ Encoding ~~~~~~~~
    # ==========================

    def encode(
        self,
        text: str,
    ):
        """
        Encode raw text into a sequence of BPE token IDs

        Steps:
        1. Pre-tokenize input text into 'words' using the regex pattern.
        2. Encode each word into bytes and map them to subword token IDs.
        3. Flatten and concatenate the resulting token IDs into a single list.

        Parameters:
            - text [str]
                The string to be tokenized.

        Returns:
            - list[int]
                List of integer token IDs representing the tokenized text.
        """
        output = []

        for word in self._pat.findall(text):
            output.extend(
                self._encode_word(
                    word.encode("utf-8")
                )
            )

        return output

    def _encode_word(self, data: bytes):
        """
        Encode a single sequence of bytes into token IDs by iteratively applying
        the learned merges in order of priority.
        """
        if not data:
            return []

        # start with the individual byte token IDs
        tokens = [self.token_to_id[bytes([b])] for b in data]

        while len(tokens) >= 2:
            # find all candidate pairs in the current token sequence
            pairs = zip(tokens, tokens[1:])

            # get the merge rank for each pair (with None if the pair was never learned)
            # find the pair with smallest rank (highest priority merge)
            pair_to_merge = min(
                pairs,
                key = lambda p: self.merge_rank.get(p, float("inf"))
            )

            # stop if no remaining adjacent pairs
            if pair_to_merge not in self.merge_rank:
                break

            # otherwise: perform the merge and update word tokenization
            new_tokens = []
            i = 0
            merged_id = self.merge_token[pair_to_merge] # retrieve merged token id based from the highest priority pair

            while i < len(tokens):
                if (
                    i < len(tokens) - 1
                    and (tokens[i], tokens[i + 1]) == pair_to_merge
                ): # merge pair!
                    new_tokens.append(merged_id)
                    i += 2  # skip past the merged pair
                else: # continue iterating
                    new_tokens.append(tokens[i])
                    i += 1

            # update tokens decomposition with merge
            tokens = new_tokens

        return tokens


    # ==========================
    # ~~~~~~~~ Decoding ~~~~~~~~
    # ==========================

    def decode(self, ids:list):
        """
        Turn list of IDs into tokens and decode UTF-8.
        """

        data = b"".join(
            self.vocab[i]
            for i in ids
        )

        return data.decode("utf-8")

In [ ]:
# Example use case

try:
  corpus = txt # your corpus here!
except:
  corpus = "The cat is on the mat. The cat is sleeping on the mat. The cat is on the sofa."


bpe = BPE()

text = "the cat sat on the mat"

print("Before Training")

ids = bpe.encode(text)

print("\nTokens:")
print(ids)

print("\nDecoded:")
print([bpe.decode([tok]) for tok in ids])

print("\nTraining the BPE model...\n")

bpe.train(
    corpus,
    vocab_size=50257, # same vocabulary size as tiktoken's gpt2 model
)

print("\nAfter Training")

ids = bpe.encode(text)

print("\nTokens:")
print(ids)

print("\nDecoded:")
print([bpe.decode([tok]) for tok in ids])